In [2]:
import requests
import pandas as pd
import os
from datetime import datetime, timedelta


def get_btc_hourly(days=365):
    #计算时间范围
    end_ts = int(datetime.now().timestamp())
    start_ts = int((datetime.now() - timedelta(days=days)).timestamp())
    url = "https://min-api.cryptocompare.com/data/v2/histohour"
    all_data = []
    current_end = end_ts
    #分批拉取数据
    while current_end > start_ts:
        #计算还需要多少小时的数据
        remaining = int((current_end - start_ts) / 3600)
        limit = min(719, remaining) 
        params = {
            "fsym": "BTC",
            "tsym": "USD",
            "limit": limit,
            "toTs": current_end
        }
        try:
            #发起请求
            resp = requests.get(url, params=params)
            resp.raise_for_status()
            data = resp.json()
            if data["Response"] == "Success":
                batch = data["Data"]["Data"]
                all_data.extend(batch)
                if batch:
                    current_end = batch[0]["time"] - 3600  # 往前推1小时
                else:
                    break
                
                #进度提示
                print(f"累计获取 {len(all_data)} 条记录...")
            else:
                print(f"API出错: {data['Message']}")
                return None
                
        except Exception as e:
            print(f"请求失败: {e}")
            return None
    #处理数据
    if all_data:
        df = pd.DataFrame(all_data)
        df["timestamp"] = pd.to_datetime(df["time"], unit="s")
        #整理需要的列
        df = df[["timestamp", "open", "high", "low", "close", "volumefrom", "volumeto"]]
        df.columns = ["timestamp", "open", "high", "low", "close", "vol_from", "vol_to"]
        #按时间排序
        return df.sort_values("timestamp").reset_index(drop=True)
    else:
        print("没拿到任何数据")
        return None
def save_csv(df, folder="btc_data"):
    if not os.path.exists(folder):
        os.makedirs(folder)
    #生成文件名
    end_date = datetime.now().strftime("%Y%m%d")
    start_date = (datetime.now() - timedelta(days=365)).strftime("%Y%m%d")
    filename = f"btc_hourly_{start_date}-{end_date}.csv"
    filepath = os.path.join(folder, filename)
    df.to_csv(filepath, index=False)
    print(f"数据已存到: {filepath}")
if __name__ == "__main__":
    #拉取最近一年的数据
    btc_df = get_btc_hourly(days=365)
    #检查数据并保存
    if btc_df is not None and not btc_df.empty:
        save_csv(btc_df)
        print("\n前5条数据:")
        print(btc_df.head())
        print(f"\n时间范围: {btc_df['timestamp'].min()} 至 {btc_df['timestamp'].max()}")
        print(f"总记录数: {len(btc_df)}")
    else:
        print("获取数据失败")


累计获取 720 条记录...
累计获取 1440 条记录...
累计获取 2160 条记录...
累计获取 2880 条记录...
累计获取 3600 条记录...
累计获取 4320 条记录...
累计获取 5040 条记录...
累计获取 5760 条记录...
累计获取 6480 条记录...
累计获取 7200 条记录...
累计获取 7920 条记录...
累计获取 8640 条记录...
累计获取 8760 条记录...
数据已存到: btc_data\btc_hourly_20240915-20250915.csv

前5条数据:
            timestamp      open      high       low     close  vol_from  \
0 2024-09-15 03:00:00  60257.18  60264.13  60159.69  60185.33    267.47   
1 2024-09-15 04:00:00  60185.33  60242.30  60103.28  60146.68    272.04   
2 2024-09-15 05:00:00  60146.68  60240.57  60145.51  60178.59    219.75   
3 2024-09-15 06:00:00  60178.59  60206.70  60160.47  60178.50     82.48   
4 2024-09-15 07:00:00  60178.50  60231.21  60146.30  60210.45    108.45   

        vol_to  
0  16106267.99  
1  16371163.47  
2  13230839.74  
3   4963151.06  
4   6528327.27  

时间范围: 2024-09-15 03:00:00 至 2025-09-15 02:00:00
总记录数: 8760
